# EDA**먼저 평가지표를 확인할 것.** 지표에 따라 `predict` / `predict_proba`가 갈리고점수 방향(높을수록 좋음 / 낮을수록 좋음)도 반대가 된다.

In [ ]:
import matplotlib.pyplot as pltimport numpy as npimport polars as pl# polars 기본 스키마 추론은 앞 100행만 본다 → 전체 스캔train = pl.read_csv('data/train.csv', infer_schema_length=None)test = pl.read_csv('data/test.csv', infer_schema_length=None)TARGET = 'target'   # TODOfeats = [c for c in train.columns if c != TARGET]train.shape, test.shape

## 1. 기본 — shape / dtype / 결측 / 값 범위

In [ ]:
print(f'train : {train.shape}')print(f'test  : {test.shape}')print(f'dtype : {sorted({str(d) for d in train.dtypes})}')print(f'결측  : train {train.null_count().to_numpy().sum()} / '      f'test {test.null_count().to_numpy().sum()}')train.head()

## 2. 타깃 분포와 기준점아무것도 안 한 제출의 점수를 미리 계산해둔다. 베이스라인이 이보다 나쁘면 버그다.

In [ ]:
dist = (train.group_by(TARGET).len()        .with_columns((pl.col('len') / train.height).alias('ratio'))        .sort(TARGET))print(dist)# 분류라면 — all-0.5 / 다수클래스 비율 제출의 LogLossy = train[TARGET].to_numpy()p = y.mean()ll = lambda q: -(y * np.log(q) + (1 - y) * np.log(1 - q)).mean()print(f'all-0.5    : {ll(0.5):.5f}')print(f'평균({p:.4f}) : {ll(p):.5f}')

## 3. 피처 타입 분포피처가 많으면 개별 분석은 무의미하다. `pl.all().n_unique()` 한 줄로 전 컬럼을 집계한다.

In [ ]:
nuniq = np.asarray(train.select(pl.all().exclude(TARGET).n_unique()).to_numpy()[0])idx = np.arange(len(feats))print(f'상수 (unique=1) : {(nuniq == 1).sum()}개')print(f'이진 (unique=2) : {(nuniq == 2).sum()}개')print(f'연속 (unique>2) : {(nuniq > 2).sum()}개')# 주의: unique=2라고 값이 {0,1}인 건 아니다. 실제 값을 확인할 것.

## 4. 상수 / 중복 피처전치하면 피처 하나가 행 하나가 되어 `is_duplicated()`로 완전 중복을 잡을 수 있다.

In [ ]:
# train/test의 같은 컬럼 dtype이 엇갈려 concat에서 SchemaError가 나는 경우가 있다allrows = pl.concat([train.drop(TARGET).cast(pl.Float64), test.cast(pl.Float64)])t = allrows.transpose(include_header=True, header_name='feature')dups = t.filter(t.drop('feature').is_duplicated())['feature'].to_list()print(f'완전 중복 피처 : {len(dups)}개  {dups[:10]}')nuniq_all = np.asarray(allrows.select(pl.all().n_unique()).to_numpy()[0])print(f'train+test 상수 : {(nuniq_all == 1).sum()}개')

## 5. 타깃과의 상관

In [ ]:
corr = np.nan_to_num(np.asarray(    train.select([pl.corr(pl.col(c), pl.col(TARGET)).alias(c) for c in feats])    .to_numpy()[0], dtype=float))order = np.argsort(corr)print('상위 10:')for i in order[::-1][:10]:    print(f'  {feats[i]:>12}  {corr[i]:+.4f}')print('하위 10:')for i in order[:10]:    print(f'  {feats[i]:>12}  {corr[i]:+.4f}')print(f'\n|corr| 최대 {np.abs(corr).max():.4f} / 평균 {np.abs(corr).mean():.4f}')

## 6. train vs test 분포 차이shift가 크면 CV를 믿기 어렵다. 작으면 LB를 자주 찔러볼 필요가 없다.

In [ ]:
X = train.select(feats).cast(pl.Float64).to_numpy()Xt = test.select(feats).cast(pl.Float64).to_numpy()gap = np.abs(X.mean(0) - Xt.mean(0)) / (X.std(0) + 1e-9)for i in np.argsort(gap)[::-1][:10]:    print(f'{feats[i]:>12}  표준화 평균차 {gap[i]:.3f}')

## 결론| 관찰 | 베이스라인 결정 ||---|---|| | |- 제거할 피처:- 모델:- 검증: